# NutriMatch Baseline Body-Fat Prediction With Enhancement Arms

This notebook tests whether body-composition traits measured about 2 years after baseline can be predicted from diet-derived features.

Targets:

- `body_comp_android_fat_free_mass` / `comp_android_fat_free_mass`
- `body_comp_total_region_percent_fat` / total region percent fat

Analyses:

- continuous regression for the baseline value
- tertile classification
- binary classification using a moving threshold selected by best AUROC in `denovo_cardiometabolic`
- clinical/exploratory high-fat classification: sex-specific high body fat for total percent fat; exploratory low android fat-free mass for android FFM

The workflow mirrors the baseline obesity notebook: run the background runner in the TRE console, then open this notebook to load saved outputs and recreate plots.

In [ ]:
from pathlib import Path
import gc
import json
import os
import re
import sys
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import pearsonr
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    mean_squared_error,
    r2_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
)
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

try:
    from lightgbm import LGBMClassifier, LGBMRegressor
    LIGHTGBM_AVAILABLE = True
except Exception:
    LGBMClassifier = None
    LGBMRegressor = None
    LIGHTGBM_AVAILABLE = False

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

RUN_TRAINING = os.environ.get('DDE_RUN_TRAINING', '0') == '1'
RANDOM_STATE = int(os.environ.get('DDE_RANDOM_STATE', '42'))
N_SPLITS = int(os.environ.get('DDE_N_SPLITS', '5'))
MIN_DAILY_KCAL = float(os.environ.get('DDE_MIN_DAILY_KCAL', '800'))
MODEL_NAME = os.environ.get('DDE_MODEL', 'lightgbm')
ALLOW_MODEL_FALLBACK = os.environ.get('DDE_ALLOW_MODEL_FALLBACK', '1') == '1'
X_BUILD_BATCH_SIZE = int(os.environ.get('DDE_X_BATCH_SIZE', '20'))
FEATURE_SET_FILTER = [x.strip() for x in os.environ.get('DDE_FEATURE_SET_FILTER', '').split(',') if x.strip()]
RUN_RF_SUPPLEMENT = os.environ.get('DDE_RUN_RF_SUPPLEMENT', '1') == '1'

print('RUN_TRAINING:', RUN_TRAINING)
print('LightGBM available:', LIGHTGBM_AVAILABLE)
print('MODEL_NAME:', MODEL_NAME)
print('Allow model fallback:', ALLOW_MODEL_FALLBACK)
print('Run RF supplement:', RUN_RF_SUPPLEMENT)
print('Minimum kcal/day filter:', MIN_DAILY_KCAL)
print('X build batch size:', X_BUILD_BATCH_SIZE)


## Paths And Feature Arms

In [ ]:
PROJECT_ROOT = Path.cwd()
tre_root = Path('/home/ec2-user/studies/Diet_Data_Enhancement_Project/Diet_Data_Enhancement_TRE')
if not (PROJECT_ROOT / 'downstream_analysis').exists() and (tre_root / 'downstream_analysis').exists():
    PROJECT_ROOT = tre_root
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

NOTEBOOK_STEM = 'nutrimatch_baseline_fat_prediction_with_enhancements'
TASK_DIR = PROJECT_ROOT / 'downstream_analysis/tasks' / NOTEBOOK_STEM
OUT_DIR = TASK_DIR / 'outputs'
FIG_DIR = OUT_DIR / 'figures'
CACHE_DIR = OUT_DIR / 'cache'
LOG_DIR = OUT_DIR / 'logs'
TRE_INPUTS = PROJECT_ROOT / 'tre_inputs'
SHARED_X_CACHE_DIR = PROJECT_ROOT / 'downstream_analysis/tasks/nutrimatch_two_year_obesity_with_enhancements/outputs/cache'
for d in [TASK_DIR, OUT_DIR, FIG_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FEATURE_SETS = [
    {'name': 'basic_nutrimatch', 'label': 'NutriMatch all nutrients', 'path': 'outputs/enhanced_hpp/2.nutrimatch_based/hpp_feature_matrix_per_100g.csv', 'feature_mode': 'enriched'},
    {'name': 'denovo_microbiome', 'label': 'De novo microbiome-oriented', 'path': 'outputs/downstream_features/denovo/microbiome/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'denovo_cardiometabolic', 'label': 'De novo cardiometabolic', 'path': 'outputs/downstream_features/denovo/cardiometabolic/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'nutrimatch_microbiome', 'label': 'NutriMatch microbiome-oriented', 'path': 'outputs/downstream_features/nutrimatch_based/microbiome/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'nutrimatch_cardiometabolic', 'label': 'NutriMatch cardiometabolic', 'path': 'outputs/downstream_features/nutrimatch_based/cardiometabolic/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'nutrimatch_broad_diet_health', 'label': 'NutriMatch broad diet-health', 'path': 'outputs/downstream_features/nutrimatch_based/broad_diet_health/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'nutrimatch_mental_health', 'label': 'NutriMatch mental-health', 'path': 'outputs/downstream_features/nutrimatch_based/mental_health/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'denovo_broad_diet_health', 'label': 'De novo broad diet-health', 'path': 'outputs/downstream_features/denovo/broad_diet_health/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'denovo_mental_health', 'label': 'De novo mental-health', 'path': 'outputs/downstream_features/denovo/mental_health/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
]
FEATURE_SETS = [fs for fs in FEATURE_SETS if (PROJECT_ROOT / fs['path']).exists()]
if FEATURE_SET_FILTER:
    wanted = set(FEATURE_SET_FILTER) | {'basic_nutrimatch'}
    FEATURE_SETS = [fs for fs in FEATURE_SETS if fs['name'] in wanted]

ARM_LABELS = {
    'age_sex_only': 'Age + sex',
    'paper_basic_nutrients': 'Age + sex + basic nutrients',
    'nutrimatch_all': 'Age + sex + NutriMatch all nutrients',
}
for fs in FEATURE_SETS:
    if fs['name'] != 'basic_nutrimatch':
        ARM_LABELS[fs['name']] = 'Age + sex + ' + fs['label']
ARM_ORDER = ['age_sex_only', 'paper_basic_nutrients', 'nutrimatch_all'] + [fs['name'] for fs in FEATURE_SETS if fs['name'] != 'basic_nutrimatch']

print('Project root:', PROJECT_ROOT)
print('Output directory:', OUT_DIR)
print('Shared X cache directory:', SHARED_X_CACHE_DIR)
print('Feature sets found:', [fs['name'] for fs in FEATURE_SETS])

## Console Runner Status

In [ ]:
def newest_tmp_log(prefix='nutrimatch_baseline_fat_prediction_with_enhancements'):
    logs = sorted(Path('/tmp').glob(prefix + '*.log'), key=lambda p: p.stat().st_mtime, reverse=True)
    return logs[0] if logs else None

latest_pid = LOG_DIR / 'background_training_latest.pid'
print('PID file exists:', latest_pid.exists())
if latest_pid.exists():
    print('PID:', latest_pid.read_text().strip())
log = newest_tmp_log()
print('Newest /tmp log:', log)
if log and log.exists():
    print(log.read_text(errors='replace')[-4000:])

## TRE Loading Helpers

In [ ]:
from downstream_analysis.data_handelling.pheno_loader_export import (
    make_loader,
    load_table_from_loader,
    dataframe_with_index_columns,
)


def read_any(path):
    path = Path(path)
    if path.suffix.lower() == '.parquet':
        return pd.read_parquet(path)
    return pd.read_csv(path, low_memory=False)


def clean_feature_name(name):
    text = str(name).lower()
    text = re.sub(r'^(enriched_daily_|enriched_|kg_weighted_|kg_|food_card_weighted_|food_card_)', '', text)
    return re.sub(r'[^a-z0-9]+', '_', text).strip('_')


def normalize_pid_series(s):
    return s.astype(str)


def find_participant_col(df):
    for col in ['participant_id', 'Participant_Study_ID', 'research_stage_id', 'user_id', 'RegistrationCode']:
        if col in df.columns:
            return col
    for col in df.columns:
        text = str(col).lower()
        if 'participant' in text or 'research_stage' in text:
            return col
    return None


def try_load_pheno_table(dataset, table=None):
    try:
        loader = make_loader(dataset, age_sex_dataset=None, errors='warn')
        table_name = table or dataset
        try:
            df = load_table_from_loader(loader, dataset, table_name, required=False)
        except Exception:
            df = None
        if df is None:
            dfs = getattr(loader, 'dfs', {})
            if table_name in dfs:
                df = dataframe_with_index_columns(dfs[table_name])
            elif len(dfs) == 1:
                df = dataframe_with_index_columns(next(iter(dfs.values())))
        return df, loader
    except Exception as exc:
        print(f'Could not load {dataset}/{table or dataset}: {exc}')
        return None, None


def find_first_matching_column(df, patterns):
    for pat in patterns:
        rx = re.compile(pat, re.IGNORECASE)
        exact = [c for c in df.columns if rx.fullmatch(str(c))]
        if exact:
            return exact[0]
        partial = [c for c in df.columns if rx.search(str(c))]
        if partial:
            return partial[0]
    return None

## Build Baseline Body Composition Targets

Baseline is the first available body-composition record for each participant. This is intentionally cross-sectional/baseline, not a future follow-up target.

In [ ]:
TARGET_SPECS = [
    {
        'target_name': 'total_region_percent_fat',
        'label': 'Total region percent fat',
        'patterns': [
            r'body_comp_total_region_percent_fat',
            r'body_composition__body_composition__body_comp_total_region_percent_fat',
            r'total.*region.*percent.*fat',
            r'total.*region.*percentage.*fat',
        ],
        'clinical': 'sex_specific_high_body_fat',
    },
    {
        'target_name': 'android_fat_free_mass',
        'label': 'Android fat-free mass',
        'patterns': [
            r'body_comp_android_fat_free_mass',
            r'body_composition__body_composition__body_comp_android_fat_free_mass',
            r'comp_android_fat_free_mass',
            r'android.*fat.*free.*mass',
            r'android.*ffm',
        ],
        'clinical': 'exploratory_low_android_ffm',
    },
]


def build_baseline_fat_targets(force_rebuild=False):
    cache = CACHE_DIR / 'baseline_fat_targets_long.csv'
    if cache.exists() and not force_rebuild:
        out = pd.read_csv(cache, low_memory=False)
        out['participant_id'] = out['participant_id'].astype(str)
        if not out.empty and {'target_value', 'target_window'}.issubset(out.columns):
            print('Loaded cached baseline targets:', cache, out.shape)
            return out
        print('Ignoring stale baseline target cache and rebuilding:', cache)

    frame, _ = try_load_pheno_table('body_composition', 'body_composition')
    if frame is None or frame.empty:
        raise FileNotFoundError('Could not load body_composition/body_composition from PhenoLoader.')

    pid_col = find_participant_col(frame)
    date_col = find_first_matching_column(frame, [r'collection_timestamp', r'collection_date', r'date', r'timestamp'])
    if pid_col is None:
        print('Body composition columns:', list(frame.columns))
        raise ValueError('Need participant column for body composition.')

    rows = []
    print('Body-composition participant column:', pid_col)
    print('Body-composition date column:', date_col)
    for spec in TARGET_SPECS:
        col = find_first_matching_column(frame, spec['patterns'])
        print(spec['target_name'], 'matched column:', col)
        if col is None:
            continue

        keep = [pid_col, col] + ([date_col] if date_col is not None else [])
        tmp = frame[keep].copy().rename(columns={pid_col: 'participant_id', col: 'target_value'})
        if date_col is not None:
            tmp = tmp.rename(columns={date_col: 'measurement_date'})
            tmp['measurement_date'] = pd.to_datetime(tmp['measurement_date'], errors='coerce')
        else:
            tmp['measurement_date'] = pd.NaT
        tmp['participant_id'] = normalize_pid_series(tmp['participant_id'])
        tmp['target_value'] = pd.to_numeric(tmp['target_value'], errors='coerce')
        tmp = tmp.dropna(subset=['participant_id', 'target_value'])
        if tmp.empty:
            continue

        sort_cols = ['participant_id'] + (['measurement_date'] if tmp['measurement_date'].notna().any() else [])
        tmp = tmp.sort_values(sort_cols).groupby('participant_id', as_index=False).first()
        tmp['target_name'] = spec['target_name']
        tmp['target_label'] = spec['label']
        tmp['source_column'] = col
        tmp['target_window'] = 'baseline_body_composition_measurement'
        rows.append(tmp[['participant_id', 'measurement_date', 'target_value', 'target_name', 'target_label', 'source_column', 'target_window']])

    if not rows:
        print('Available body-composition columns containing fat/region/android:')
        for col in frame.columns:
            if re.search(r'fat|region|android|percent', str(col), re.IGNORECASE):
                print(' ', col)
        raise ValueError('No baseline fat targets could be built. Check body composition column names.')

    out = pd.concat(rows, ignore_index=True)
    out.to_csv(cache, index=False)
    print('Wrote baseline targets:', cache, out.shape)
    return out

targets_long = build_baseline_fat_targets()
display(targets_long.groupby(['target_name', 'target_label', 'source_column', 'target_window']).agg(n=('participant_id', 'nunique'), mean=('target_value', 'mean'), sd=('target_value', 'std')).reset_index())
display(targets_long.head())


## Build Covariates

In [ ]:
def load_covariates(participant_ids):
    cache = CACHE_DIR / 'baseline_fat_covariates.csv'
    if cache.exists():
        cov = pd.read_csv(cache, low_memory=False)
        cov['participant_id'] = cov['participant_id'].astype(str)
        print('Loaded cached covariates:', cache, cov.shape)
        return cov
    specs = [('body_composition', 'age_sex'), ('anthropometrics', 'age_sex'), ('population', 'population')]
    frames = []
    cov_rx = re.compile(r'^age$|age_at|sex$|gender$|year_of_birth', re.IGNORECASE)
    for dataset, table in specs:
        frame, _ = try_load_pheno_table(dataset, table)
        if frame is None:
            continue
        pid_col = find_participant_col(frame)
        if pid_col is None:
            continue
        matches = [c for c in frame.columns if cov_rx.search(str(c))]
        if not matches:
            continue
        part = frame[[pid_col] + matches].copy().rename(columns={pid_col: 'participant_id'})
        part['participant_id'] = part['participant_id'].astype(str)
        rename = {}
        for c in matches:
            lc = str(c).lower()
            if 'sex' in lc or 'gender' in lc:
                rename[c] = 'sex'
            elif 'year_of_birth' in lc:
                rename[c] = 'year_of_birth'
            elif 'age' in lc:
                rename[c] = 'age'
        part = part.rename(columns=rename)
        keep = ['participant_id'] + [c for c in ['age', 'sex', 'year_of_birth'] if c in part.columns]
        part = part[keep]
        if 'age' in part.columns:
            part['age'] = pd.to_numeric(part['age'], errors='coerce')
        if 'year_of_birth' in part.columns and 'age' not in part.columns:
            part['year_of_birth'] = pd.to_numeric(part['year_of_birth'], errors='coerce')
            part['age'] = 2022 - part['year_of_birth']
            part = part.drop(columns=['year_of_birth'])
        elif 'year_of_birth' in part.columns:
            part = part.drop(columns=['year_of_birth'])
        frames.append(part.groupby('participant_id', as_index=False).first())
    cov = pd.DataFrame({'participant_id': pd.Series(participant_ids).astype(str).unique()})
    for part in frames:
        for col in [c for c in part.columns if c != 'participant_id']:
            if col not in cov.columns:
                cov = cov.merge(part[['participant_id', col]], on='participant_id', how='left')
            else:
                add = part[['participant_id', col]].rename(columns={col: f'{col}_new'})
                cov = cov.merge(add, on='participant_id', how='left')
                cov[col] = cov[col].combine_first(cov[f'{col}_new'])
                cov = cov.drop(columns=[f'{col}_new'])
    if 'sex' in cov.columns:
        sex_text = cov['sex'].astype(str).str.lower()
        cov['sex_norm'] = np.select(
            [sex_text.str.startswith('m') | sex_text.isin(['1', 'male']), sex_text.str.startswith('f') | sex_text.isin(['0', '2', 'female'])],
            ['male', 'female'],
            default=np.nan,
        )
    cov.to_csv(cache, index=False)
    print('Wrote covariates:', cache, cov.shape)
    return cov

covariates = load_covariates(targets_long['participant_id'])
print('Covariate non-null counts:')
display(covariates.notna().sum())
display(covariates.head())

## Build Or Reuse 800-kcal Diet Features

In [ ]:
PAPER_BASIC_NUTRIENT_NAMES = ['Energy', 'Protein', 'Total lipid (fat)', 'Carbohydrate, by difference', 'Fiber, total dietary', 'Sodium, Na', 'Water', 'Alcohol, ethyl']
BASIC_NUTRIENT_KEYS = {clean_feature_name(x) for x in PAPER_BASIC_NUTRIENT_NAMES}
PAPER_BASIC_PATTERNS = {
    'energy': re.compile(r'(^|_)energy($|_)|calorie|kcal', re.IGNORECASE),
    'protein': re.compile(r'(^|_)protein($|_)', re.IGNORECASE),
    'total_lipid_fat': re.compile(r'total_lipid|lipid|total_fat|(^|_)fat($|_)', re.IGNORECASE),
    'carbohydrate_by_difference': re.compile(r'carbohydrate|(^|_)carb($|_)', re.IGNORECASE),
    'fiber_total_dietary': re.compile(r'fiber|fibre', re.IGNORECASE),
    'sodium_na': re.compile(r'sodium|(^|_)na($|_)', re.IGNORECASE),
    'water': re.compile(r'(^|_)water($|_)', re.IGNORECASE),
    'alcohol_ethyl': re.compile(r'alcohol|ethyl', re.IGNORECASE),
}


def paper_basic_nutrient_kind(col):
    name = clean_feature_name(col)
    if name in BASIC_NUTRIENT_KEYS:
        return name
    for kind, pattern in PAPER_BASIC_PATTERNS.items():
        if pattern.search(name):
            return kind
    return None


def load_diet_events():
    for path in [TRE_INPUTS / 'diet_logging_events.parquet', TRE_INPUTS / 'diet_logging_events.csv']:
        if path.exists():
            print('Loading diet events:', path)
            return read_any(path)
    frame, _ = try_load_pheno_table('diet_logging', 'diet_logging_events')
    if frame is None:
        raise FileNotFoundError('Could not load diet_logging_events.')
    TRE_INPUTS.mkdir(parents=True, exist_ok=True)
    out = TRE_INPUTS / 'diet_logging_events.csv'
    frame.to_csv(out, index=False)
    return frame


def choose_day_col(df):
    if 'logging_day' in df.columns:
        return 'logging_day'
    for c in ['collection_date', 'local_date', 'date']:
        if c in df.columns:
            return c
    for c in ['collection_timestamp', 'local_timestamp', 'timestamp']:
        if c in df.columns:
            return c
    return None


def build_filtered_participant_food():
    own_cache = CACHE_DIR / f'diet_participant_food_gef_{int(MIN_DAILY_KCAL)}kcal.csv'
    own_days = CACHE_DIR / f'diet_valid_days_gef_{int(MIN_DAILY_KCAL)}kcal.csv'
    upstream_cache = SHARED_X_CACHE_DIR / f'diet_participant_food_gef_{int(MIN_DAILY_KCAL)}kcal.csv'
    upstream_days = SHARED_X_CACHE_DIR / f'diet_valid_days_gef_{int(MIN_DAILY_KCAL)}kcal.csv'
    for cache, days_cache in [(own_cache, own_days), (upstream_cache, upstream_days)]:
        if cache.exists() and days_cache.exists():
            diet = pd.read_csv(cache, low_memory=False)
            valid_days = pd.read_csv(days_cache, low_memory=False)
            diet['participant_id'] = diet['participant_id'].astype(str)
            valid_days['participant_id'] = valid_days['participant_id'].astype(str)
            print('Loaded cached filtered diet:', cache, diet.shape)
            return diet, valid_days
    events = load_diet_events()
    required = ['participant_id', 'food_id', 'weight_g']
    missing = [c for c in required if c not in events.columns]
    if missing:
        raise ValueError(f'Diet events missing required columns: {missing}')
    day_col = choose_day_col(events)
    if day_col is None:
        events['_diet_day'] = 'all_days'
    else:
        events['_diet_day'] = events[day_col]
        if day_col != 'logging_day':
            events['_diet_day'] = pd.to_datetime(events['_diet_day'], errors='coerce').dt.date.astype(str)
    events['participant_id'] = events['participant_id'].astype(str)
    events['food_id'] = events['food_id'].astype(str)
    events['weight_g'] = pd.to_numeric(events['weight_g'], errors='coerce').fillna(0.0)
    kcal_col = find_first_matching_column(events, [r'calories_kcal', r'energy_kcal', r'kcal', r'calorie'])
    if kcal_col is not None:
        events['_kcal'] = pd.to_numeric(events[kcal_col], errors='coerce').fillna(0.0)
        day_energy = events.groupby(['participant_id', '_diet_day'], as_index=False)['_kcal'].sum()
        valid_day_keys = day_energy[day_energy['_kcal'] >= MIN_DAILY_KCAL][['participant_id', '_diet_day']]
        events = events.merge(valid_day_keys, on=['participant_id', '_diet_day'], how='inner')
        print('Applied 800 kcal/day filter using', kcal_col, '| valid days:', len(valid_day_keys), 'of', len(day_energy))
    else:
        print('No calorie column found; aggregating all days without kcal filter.')
    valid_days = events.groupby('participant_id', as_index=False)['_diet_day'].nunique().rename(columns={'_diet_day': 'valid_diet_days'})
    diet = events.groupby(['participant_id', 'food_id'], as_index=False)['weight_g'].sum()
    diet = diet.merge(valid_days, on='participant_id', how='left')
    diet.to_csv(own_cache, index=False)
    valid_days.to_csv(own_days, index=False)
    print('Wrote filtered diet:', own_cache, diet.shape)
    return diet, valid_days

participant_food, valid_days = build_filtered_participant_food()
display(valid_days['valid_diet_days'].describe())

In [ ]:
def choose_ref_food_col(ref):
    for col in ['hpp_food_id', 'food_id']:
        if col in ref.columns:
            return col
    raise ValueError('Could not find hpp_food_id or food_id in feature table')


def feature_columns(ref, ref_food_col, feature_mode):
    if feature_mode == 'embedding':
        cols = [c for c in ref.columns if str(c).startswith('embedding_')]
    else:
        exclude = {ref_food_col, 'food_id', 'hpp_food_id', 'food_name', 'short_food_name', 'product_name'}
        cols = [c for c in ref.columns if c not in exclude and pd.api.types.is_numeric_dtype(ref[c])]
    return list(dict.fromkeys(cols))


def x_cache_candidates(fs_name):
    fname = f'X_{fs_name}_participant_gef_{int(MIN_DAILY_KCAL)}kcal.csv'
    return [CACHE_DIR / fname, SHARED_X_CACHE_DIR / fname]


def build_participant_x(fs, batch_size=None):
    if batch_size is None:
        batch_size = X_BUILD_BATCH_SIZE
    for path in x_cache_candidates(fs['name']):
        if path.exists():
            x = pd.read_csv(path, low_memory=False)
            x['participant_id'] = x['participant_id'].astype(str)
            print('Loaded cached X:', path, x.shape)
            return x
    print('Building X:', fs['name'], '| batch_size=', batch_size, flush=True)
    ref = read_any(PROJECT_ROOT / fs['path'])
    ref_food_col = choose_ref_food_col(ref)
    cols = feature_columns(ref, ref_food_col, fs['feature_mode'])
    if not cols:
        raise ValueError(f"No numeric feature columns found for {fs['name']}")
    ref = ref[[ref_food_col] + cols].copy()
    ref['_food_join_id'] = ref[ref_food_col].astype(str)
    diet = participant_food.copy()
    diet['_food_join_id'] = diet['food_id'].astype(str)
    total_grams = diet.groupby('participant_id')['weight_g'].sum().replace(0, np.nan)
    days = valid_days.set_index('participant_id')['valid_diet_days'].replace(0, np.nan)
    parts = []
    for start in range(0, len(cols), batch_size):
        batch = cols[start:start + batch_size]
        print('  feature batch', start + 1, '-', min(start + batch_size, len(cols)), 'of', len(cols), flush=True)
        merged = diet[['participant_id', '_food_join_id', 'weight_g']].merge(ref[['_food_join_id'] + batch], on='_food_join_id', how='left')
        values = merged[batch].apply(pd.to_numeric, errors='coerce').fillna(0.0)
        if fs['feature_mode'] == 'enriched':
            scaled = values.mul(merged['weight_g'].to_numpy() / 100.0, axis=0)
            agg = scaled.assign(participant_id=merged['participant_id'].values).groupby('participant_id').sum(numeric_only=True)
            agg = agg.div(days, axis=0).fillna(0.0)
            prefix = 'enriched_daily_'
        elif fs['feature_mode'] in ['kg', 'embedding']:
            scaled = values.mul(merged['weight_g'].to_numpy(), axis=0)
            agg = scaled.assign(participant_id=merged['participant_id'].values).groupby('participant_id').sum(numeric_only=True)
            agg = agg.div(total_grams, axis=0).fillna(0.0)
            prefix = 'kg_weighted_' if fs['feature_mode'] == 'kg' else 'food_card_weighted_'
        else:
            raise ValueError(fs['feature_mode'])
        agg.columns = [prefix + str(c) for c in agg.columns]
        parts.append(agg)
        del merged, values, scaled, agg
        gc.collect()
    x = pd.concat(parts, axis=1).reset_index()
    out = CACHE_DIR / f"X_{fs['name']}_participant_gef_{int(MIN_DAILY_KCAL)}kcal.csv"
    x.to_csv(out, index=False)
    print('Wrote X:', out, x.shape, flush=True)
    del ref, diet, total_grams, days, parts
    gc.collect()
    return x

x_tables = {}
for fs in FEATURE_SETS:
    x_tables[fs['name']] = build_participant_x(fs)
    gc.collect()
print('Built/loaded X tables:', {k: v.shape for k, v in x_tables.items()})

## Create Model Arms

In [ ]:
basic_x = x_tables['basic_nutrimatch']
all_basic_feature_cols = [c for c in basic_x.columns if c != 'participant_id' and pd.api.types.is_numeric_dtype(basic_x[c])]
paper_basic_cols_by_kind = {}
for col in all_basic_feature_cols:
    kind = paper_basic_nutrient_kind(col)
    if kind and kind not in paper_basic_cols_by_kind:
        paper_basic_cols_by_kind[kind] = col
paper_basic_cols = list(paper_basic_cols_by_kind.values())
print('Paper-basic nutrient kinds found:', sorted(paper_basic_cols_by_kind))
print('Paper-basic nutrient columns found:', paper_basic_cols)
if not paper_basic_cols:
    raise ValueError('No paper-basic nutrient columns were found.')

covariate_cols = [c for c in ['age', 'sex'] if c in covariates.columns and covariates[c].notna().any()]
print('Using covariates:', covariate_cols)

arms = [
    {'arm': 'age_sex_only', 'label': ARM_LABELS['age_sex_only'], 'x': covariates[['participant_id'] + covariate_cols].copy()},
    {'arm': 'paper_basic_nutrients', 'label': ARM_LABELS['paper_basic_nutrients'], 'x': basic_x[['participant_id'] + paper_basic_cols].merge(covariates[['participant_id'] + covariate_cols], on='participant_id', how='left')},
    {'arm': 'nutrimatch_all', 'label': ARM_LABELS['nutrimatch_all'], 'x': basic_x.merge(covariates[['participant_id'] + covariate_cols], on='participant_id', how='left')},
]
for fs in FEATURE_SETS:
    if fs['name'] == 'basic_nutrimatch':
        continue
    x = x_tables[fs['name']].merge(covariates[['participant_id'] + covariate_cols], on='participant_id', how='left')
    arms.append({'arm': fs['name'], 'label': ARM_LABELS[fs['name']], 'x': x})

for arm in arms:
    print(arm['arm'], arm['x'].shape, arm['label'])

## Build Continuous And Categorical Tasks

In [ ]:
def qcut_safe(values, q, labels):
    try:
        return pd.qcut(values, q=q, labels=labels, duplicates='drop')
    except Exception:
        return pd.Series(index=values.index, dtype='object')


def make_task_table():
    rows = []
    threshold_rows = []
    for spec in TARGET_SPECS:
        name = spec['target_name']
        part = targets_long[targets_long['target_name'].eq(name)].copy()
        if part.empty:
            continue
        base = part[['participant_id', 'target_value']].copy()
        base['participant_id'] = base['participant_id'].astype(str)
        base = base.merge(covariates[['participant_id', 'sex_norm']] if 'sex_norm' in covariates.columns else covariates[['participant_id']], on='participant_id', how='left')
        rows.append({'target_name': name, 'target_label': spec['label'], 'task_id': f'{name}__continuous', 'task_family': 'continuous', 'task_type': 'regression', 'threshold': np.nan, 'threshold_note': '', 'data': base[['participant_id', 'target_value']]})
        tert = base.copy()
        tert['target_value'] = qcut_safe(tert['target_value'], 3, labels=[0, 1, 2]).astype('float')
        rows.append({'target_name': name, 'target_label': spec['label'], 'task_id': f'{name}__tertile', 'task_family': 'tertile', 'task_type': 'classification', 'threshold': np.nan, 'threshold_note': 'tertiles of baseline body-composition value', 'data': tert[['participant_id', 'target_value']]})
        if name == 'total_region_percent_fat':
            clin = base.copy()
            if 'sex_norm' in clin.columns:
                clin['target_value'] = np.where(
                    clin['sex_norm'].eq('male'),
                    (clin['target_value'] >= 25.0).astype(int),
                    np.where(clin['sex_norm'].eq('female'), (clin['target_value'] >= 35.0).astype(int), np.nan),
                )
                note = 'high body fat: male >=25%, female >=35%'
            else:
                clin['target_value'] = (clin['target_value'] >= 30.0).astype(int)
                note = 'high body fat: >=30%, sex not available'
            rows.append({'target_name': name, 'target_label': spec['label'], 'task_id': f'{name}__clinical_high_fat', 'task_family': 'clinical_high_fat', 'task_type': 'classification', 'threshold': np.nan, 'threshold_note': note, 'data': clin[['participant_id', 'target_value']]})
        else:
            exploratory_cut = float(base['target_value'].quantile(1/3))
            clin = base.copy()
            clin['target_value'] = (clin['target_value'] <= exploratory_cut).astype(int)
            rows.append({'target_name': name, 'target_label': spec['label'], 'task_id': f'{name}__exploratory_low_ffm', 'task_family': 'exploratory_low_ffm', 'task_type': 'classification', 'threshold': exploratory_cut, 'threshold_note': 'exploratory low android fat-free mass: bottom tertile', 'data': clin[['participant_id', 'target_value']]})
        for q in np.linspace(0.2, 0.8, 13):
            threshold = float(base['target_value'].quantile(q))
            if name == 'android_fat_free_mass':
                y = (base['target_value'] <= threshold).astype(int)
                direction = '<='
            else:
                y = (base['target_value'] >= threshold).astype(int)
                direction = '>='
            counts = y.value_counts()
            if y.nunique() == 2 and counts.min() >= N_SPLITS:
                threshold_rows.append({'target_name': name, 'target_label': spec['label'], 'quantile': q, 'threshold': threshold, 'direction': direction, 'positive_rate': float(y.mean())})
    threshold_candidates = pd.DataFrame(threshold_rows)
    return rows, threshold_candidates

task_rows, threshold_candidates = make_task_table()
threshold_candidates.to_csv(OUT_DIR / 'baseline_fat_threshold_candidates.csv', index=False)
print('Initial tasks:', [(r['task_id'], r['task_type']) for r in task_rows])
display(threshold_candidates)

## Model Evaluation Helpers

In [ ]:
def make_onehot():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


def model_for(task_type, model_name=MODEL_NAME):
    if model_name == 'lightgbm':
        if LIGHTGBM_AVAILABLE:
            if task_type == 'classification':
                return LGBMClassifier(n_estimators=300, learning_rate=0.03, random_state=RANDOM_STATE, verbose=-1)
            return LGBMRegressor(n_estimators=300, learning_rate=0.03, random_state=RANDOM_STATE, verbose=-1)
        if not ALLOW_MODEL_FALLBACK:
            raise ImportError('LightGBM is required but unavailable.')
        print('LightGBM requested but unavailable; falling back to HistGradientBoosting.', flush=True)
        model_name = 'hist_gradient_boosting'
    if model_name == 'hist_gradient_boosting':
        if task_type == 'classification':
            return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.03, random_state=RANDOM_STATE)
        return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.03, random_state=RANDOM_STATE)
    if model_name == 'random_forest':
        if task_type == 'classification':
            return RandomForestClassifier(n_estimators=500, max_features='sqrt', class_weight='balanced_subsample', n_jobs=-1, random_state=RANDOM_STATE)
        return RandomForestRegressor(n_estimators=500, max_features='sqrt', n_jobs=-1, random_state=RANDOM_STATE)
    raise ValueError(model_name)


def effective_model_name(model_name=MODEL_NAME):
    if model_name == 'lightgbm' and not LIGHTGBM_AVAILABLE and ALLOW_MODEL_FALLBACK:
        return 'hist_gradient_boosting_fallback'
    return model_name


def pipeline_for(X, task_type, model_name=MODEL_NAME):
    numeric_cols = X.select_dtypes(include=[np.number, bool]).columns.tolist()
    cat_cols = [c for c in X.columns if c not in numeric_cols]
    pre = ColumnTransformer(
        transformers=[
            ('num', Pipeline([('impute', SimpleImputer())]), numeric_cols),
            ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', make_onehot())]), cat_cols),
        ],
        remainder='drop',
    )
    return Pipeline([('pre', pre), ('model', model_for(task_type, model_name))])


def classification_scores(y_test, pred, est, X_test):
    """Return fold metrics and a scalar OOF score.

    Binary tasks get AUROC/AUPRC against class 1 when present. Tertile tasks get
    macro one-vs-rest AUROC when the estimator exposes class probabilities; AUPRC
    is left blank because there is no single paper-style PR curve for multiclass.
    """
    proba = None
    if hasattr(est, 'predict_proba'):
        proba = est.predict_proba(X_test)
    classes = getattr(est.named_steps.get('model'), 'classes_', np.sort(pd.Series(y_test).unique()))
    classes = list(classes)

    auroc = np.nan
    auprc = np.nan
    if proba is not None and proba.ndim == 2 and len(classes) == 2:
        positive_class = 1 if 1 in classes else classes[-1]
        positive_idx = classes.index(positive_class)
        score = proba[:, positive_idx]
        auroc = float(roc_auc_score(y_test, score))
        auprc = float(average_precision_score(y_test, score))
    elif proba is not None and proba.ndim == 2 and len(classes) > 2:
        score = proba.max(axis=1)
        try:
            auroc = float(roc_auc_score(y_test, proba, multi_class='ovr', average='macro'))
        except Exception:
            auroc = np.nan
    else:
        raw = est.decision_function(X_test)
        score = np.asarray(raw)
        if score.ndim > 1:
            score = score.max(axis=1)
        if pd.Series(y_test).nunique() == 2:
            auroc = float(roc_auc_score(y_test, score))
            auprc = float(average_precision_score(y_test, score))

    row = {
        'auroc': auroc,
        'auprc': auprc,
        'accuracy': float(accuracy_score(y_test, pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_test, pred)),
        'f1_macro': float(f1_score(y_test, pred, average='macro', zero_division=0)),
    }
    return row, np.asarray(score, dtype=float)


def evaluate_arm(arm, y_table, task_type, model_name=MODEL_NAME):
    y_table = y_table.copy()
    y_table['participant_id'] = y_table['participant_id'].astype(str)
    merged = arm['x'].merge(y_table, on='participant_id', how='inner')
    merged = merged.dropna(subset=['target_value'])
    participants = merged['participant_id'].copy()
    X = merged.drop(columns=['participant_id', 'target_value'])
    y = merged['target_value']
    if task_type == 'classification':
        y = y.astype(int)
        counts = y.value_counts()
        if y.nunique() < 2 or counts.min() < N_SPLITS:
            return None, pd.DataFrame()
        cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
        split_iter = cv.split(X, y)
    else:
        y = pd.to_numeric(y, errors='coerce')
        valid = y.notna()
        X, y, participants = X.loc[valid], y.loc[valid], participants.loc[valid]
        if len(y) < N_SPLITS:
            return None, pd.DataFrame()
        cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
        split_iter = cv.split(X)
    estimator = pipeline_for(X, task_type, model_name)
    fold_rows = []
    oof_rows = []
    for fold, (train_idx, test_idx) in enumerate(split_iter, start=1):
        est = clone(estimator)
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        est.fit(X_train, y_train)
        pred = est.predict(X_test)
        if task_type == 'classification':
            row, score = classification_scores(y_test, pred, est, X_test)
            row['fold'] = fold
            fold_rows.append(row)
            for pid, yt, yp, sc in zip(participants.iloc[test_idx], y_test, pred, score):
                oof_rows.append({'participant_id': pid, 'fold': fold, 'y_true': int(yt), 'y_pred': int(yp), 'y_score': float(sc)})
        else:
            try:
                pr = float(pearsonr(y_test, pred).statistic)
            except Exception:
                pr = np.nan
            fold_rows.append({'fold': fold, 'r2': float(r2_score(y_test, pred)), 'rmse': float(np.sqrt(mean_squared_error(y_test, pred))), 'pearson_r': pr})
            for pid, yt, yp in zip(participants.iloc[test_idx], y_test, pred):
                oof_rows.append({'participant_id': pid, 'fold': fold, 'y_true': float(yt), 'y_pred': float(yp), 'y_score': np.nan})
    fold_df = pd.DataFrame(fold_rows)
    oof = pd.DataFrame(oof_rows)
    summary = {'arm': arm['arm'], 'label': arm['label'], 'model': effective_model_name(model_name), 'task_type': task_type, 'n': int(len(y)), 'feature_count': int(X.shape[1])}
    for col in fold_df.columns:
        if col == 'fold':
            continue
        summary[f'{col}_mean'] = float(fold_df[col].mean())
        summary[f'{col}_std'] = float(fold_df[col].std())
    if task_type == 'regression' and not oof.empty:
        summary['oof_r2'] = float(r2_score(oof['y_true'], oof['y_pred']))
    if task_type == 'classification' and not oof.empty and oof['y_true'].nunique() == 2:
        summary['oof_auroc'] = float(roc_auc_score(oof['y_true'], oof['y_score']))
        summary['oof_auprc'] = float(average_precision_score(oof['y_true'], oof['y_score']))
    oof['arm'] = arm['arm']
    oof['label'] = arm['label']
    return summary, oof


## Select Moving Thresholds Using De Novo Cardiometabolic

In [ ]:
def scan_best_thresholds():
    scan_path = OUT_DIR / 'baseline_fat_denovo_cardiometabolic_threshold_scan.csv'
    best_path = OUT_DIR / 'baseline_fat_best_moving_thresholds.csv'
    if not RUN_TRAINING and scan_path.exists() and best_path.exists():
        return pd.read_csv(scan_path), pd.read_csv(best_path)
    if threshold_candidates.empty:
        return pd.DataFrame(), pd.DataFrame()
    arm_lookup = {a['arm']: a for a in arms}
    scan_rows = []
    for _, cand in threshold_candidates.iterrows():
        name = cand['target_name']
        part = targets_long[targets_long['target_name'].eq(name)][['participant_id', 'target_value']].rename(columns={'target_value': 'target_value'}).copy()
        if cand['direction'] == '<=':
            part['target_value'] = (part['target_value'] <= cand['threshold']).astype(int)
        else:
            part['target_value'] = (part['target_value'] >= cand['threshold']).astype(int)
        metrics, _ = evaluate_arm(arm_lookup['denovo_cardiometabolic'], part[['participant_id', 'target_value']], 'classification', model_name=MODEL_NAME)
        if metrics is None:
            continue
        scan_rows.append({**cand.to_dict(), 'arm_used_for_selection': 'denovo_cardiometabolic', 'selection_auroc': metrics.get('auroc_mean'), 'selection_auprc': metrics.get('auprc_mean')})
        print('threshold scan', name, cand['direction'], cand['threshold'], 'AUROC', metrics.get('auroc_mean'), flush=True)
    scan = pd.DataFrame(scan_rows)
    best = scan.sort_values('selection_auroc', ascending=False).groupby('target_name', as_index=False).head(1) if not scan.empty else pd.DataFrame()
    scan.to_csv(scan_path, index=False)
    best.to_csv(best_path, index=False)
    return scan, best

threshold_scan, best_thresholds = scan_best_thresholds()
print('Best moving thresholds:')
display(best_thresholds)

In [ ]:
# Add moving-threshold tasks after threshold selection.
for _, row in best_thresholds.iterrows():
    name = row['target_name']
    part = targets_long[targets_long['target_name'].eq(name)][['participant_id', 'target_value']].rename(columns={'target_value': 'target_value'}).copy()
    if row['direction'] == '<=':
        part['target_value'] = (part['target_value'] <= row['threshold']).astype(int)
    else:
        part['target_value'] = (part['target_value'] >= row['threshold']).astype(int)
    task_rows.append({
        'target_name': name,
        'target_label': row['target_label'],
        'task_id': f'{name}__best_denovo_cardiometabolic_threshold',
        'task_family': 'best_denovo_cardiometabolic_threshold',
        'task_type': 'classification',
        'threshold': row['threshold'],
        'threshold_note': f"selected by denovo_cardiometabolic AUROC; positive if value {row['direction']} {row['threshold']:.4g}",
        'data': part[['participant_id', 'target_value']],
    })

task_catalog = pd.DataFrame([{k: v for k, v in row.items() if k != 'data'} for row in task_rows])
task_catalog.to_csv(OUT_DIR / 'baseline_fat_task_catalog.csv', index=False)
display(task_catalog)

## Train And Evaluate All Tasks

In [ ]:
results_path = OUT_DIR / 'baseline_fat_model_results.csv'
oof_path = OUT_DIR / 'baseline_fat_oof_predictions.csv'

if RUN_TRAINING:
    result_rows = []
    oof_all = []
    for task in task_rows:
        print('\nTask:', task['task_id'], task['task_type'], task['threshold_note'], flush=True)
        for arm in arms:
            print('  Training:', arm['arm'], flush=True)
            metrics, oof = evaluate_arm(arm, task['data'], task['task_type'], model_name=MODEL_NAME)
            if metrics is None:
                print('    skipped', flush=True)
                continue
            row = {k: v for k, v in task.items() if k != 'data'}
            row.update(metrics)
            result_rows.append(row)
            oof['target_name'] = task['target_name']
            oof['target_label'] = task['target_label']
            oof['task_id'] = task['task_id']
            oof['task_family'] = task['task_family']
            oof['task_type'] = task['task_type']
            oof_all.append(oof)
            key = metrics.get('r2_mean') if task['task_type'] == 'regression' else metrics.get('auroc_mean')
            print('    primary fold metric:', key, flush=True)
    results = pd.DataFrame(result_rows)
    oof_predictions = pd.concat(oof_all, ignore_index=True) if oof_all else pd.DataFrame()
    results.to_csv(results_path, index=False)
    oof_predictions.to_csv(oof_path, index=False)
    print('Wrote:', results_path, results.shape)
    print('Wrote:', oof_path, oof_predictions.shape)
elif results_path.exists():
    results = pd.read_csv(results_path, low_memory=False)
    oof_predictions = pd.read_csv(oof_path, low_memory=False) if oof_path.exists() else pd.DataFrame()
    print('Loaded saved results:', results_path, results.shape)
else:
    results = pd.DataFrame()
    oof_predictions = pd.DataFrame()
    print('Training skipped and no saved results found. Run the background runner first.')

def add_primary_metric(df):
    df = df.copy()
    df['primary_metric_name'] = np.where(df['task_type'].eq('regression'), 'R2', 'AUROC')
    df['primary_metric'] = np.nan
    if 'r2_mean' in df.columns:
        df.loc[df['task_type'].eq('regression'), 'primary_metric'] = df.loc[df['task_type'].eq('regression'), 'r2_mean']
    if 'auroc_mean' in df.columns:
        df.loc[df['task_type'].eq('classification'), 'primary_metric'] = df.loc[df['task_type'].eq('classification'), 'auroc_mean']
    return df

results = add_primary_metric(results) if not results.empty else results
display(results.sort_values(['task_id', 'primary_metric'], ascending=[True, False]).head(120) if not results.empty else results)


## Paper-Ready Tables

In [ ]:
if not results.empty:
    metric_table = results.copy()
    metric_table.to_csv(OUT_DIR / 'baseline_fat_metric_table.csv', index=False)
    arm_summary = results.groupby(['target_name', 'target_label', 'task_id', 'task_family', 'task_type', 'arm', 'label', 'model'], as_index=False).agg(
        n=('n', 'max'),
        feature_count=('feature_count', 'max'),
        mean_r2=('r2_mean', 'mean'),
        mean_rmse=('rmse_mean', 'mean'),
        mean_pearson_r=('pearson_r_mean', 'mean'),
        mean_auroc=('auroc_mean', 'mean'),
        mean_auprc=('auprc_mean', 'mean'),
        mean_accuracy=('accuracy_mean', 'mean'),
        mean_balanced_accuracy=('balanced_accuracy_mean', 'mean'),
        mean_f1_macro=('f1_macro_mean', 'mean'),
        mean_primary_metric=('primary_metric', 'mean'),
    ).sort_values(['task_id', 'mean_primary_metric'], ascending=[True, False])
    arm_summary.to_csv(OUT_DIR / 'baseline_fat_arm_summary.csv', index=False)

    paper_table = results[[c for c in [
        'target_name', 'target_label', 'task_id', 'task_family', 'task_type', 'threshold', 'threshold_note',
        'arm', 'label', 'model', 'n', 'feature_count', 'r2_mean', 'r2_std', 'oof_r2', 'rmse_mean', 'pearson_r_mean',
        'auroc_mean', 'auroc_std', 'oof_auroc', 'auprc_mean', 'oof_auprc', 'accuracy_mean', 'balanced_accuracy_mean', 'f1_macro_mean',
        'primary_metric_name', 'primary_metric'
    ] if c in results.columns]].sort_values(['task_id', 'primary_metric'], ascending=[True, False])
    paper_table.to_csv(OUT_DIR / 'paper_ready_baseline_fat_table.csv', index=False)
    best_by_task = paper_table.groupby(['target_name', 'task_id', 'task_type'], as_index=False).head(1)
    best_by_task.to_csv(OUT_DIR / 'baseline_fat_best_by_task.csv', index=False)
    print('Wrote:', OUT_DIR / 'baseline_fat_arm_summary.csv')
    print('Wrote:', OUT_DIR / 'paper_ready_baseline_fat_table.csv')
    print('Wrote:', OUT_DIR / 'baseline_fat_best_by_task.csv')
    display(paper_table)
else:
    paper_table = pd.DataFrame()
    best_by_task = pd.DataFrame()
    print('No results loaded.')


## Regression Plots

In [ ]:
def save_fig(path):
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=240, bbox_inches='tight')
    print('Wrote:', path)

if not results.empty:
    reg = results[results['task_type'].eq('regression')].copy()
    if not reg.empty:
        plt.figure(figsize=(11, max(5, 0.42 * reg['label'].nunique())))
        sns.barplot(data=reg.sort_values('r2_mean', ascending=False), y='label', x='r2_mean', hue='target_label', orient='h')
        plt.axvline(0, color='black', lw=0.8)
        plt.title('baseline body-composition regression R2 by feature set')
        plt.xlabel('Mean fold R2')
        plt.ylabel('')
        save_fig(FIG_DIR / 'baseline_fat_regression_r2_by_feature_set.png')
        plt.show()

        for target_name in reg['target_name'].unique():
            target_reg = reg[reg['target_name'].eq(target_name)]
            best_enh = target_reg[~target_reg['arm'].isin(['age_sex_only', 'paper_basic_nutrients', 'nutrimatch_all'])].sort_values('r2_mean', ascending=False).head(1)
            compare_arms = ['nutrimatch_all'] + (best_enh['arm'].tolist() if not best_enh.empty else [])
            fig, axes = plt.subplots(1, len(compare_arms), figsize=(6 * len(compare_arms), 5), sharex=False, sharey=False)
            if not isinstance(axes, np.ndarray):
                axes = np.array([axes])
            for ax, arm_name in zip(axes, compare_arms):
                p = oof_predictions[(oof_predictions['target_name'].eq(target_name)) & (oof_predictions['task_type'].eq('regression')) & (oof_predictions['arm'].eq(arm_name))]
                if p.empty:
                    ax.set_axis_off()
                    continue
                sns.scatterplot(data=p, x='y_true', y='y_pred', ax=ax, s=18, alpha=0.45)
                sns.regplot(data=p, x='y_true', y='y_pred', ax=ax, scatter=False, color='black')
                ax.set_title(ARM_LABELS.get(arm_name, arm_name))
                ax.set_xlabel('Observed baseline value')
                ax.set_ylabel('Predicted baseline value')
            save_fig(FIG_DIR / f'baseline_fat_predicted_vs_observed_{target_name}.png')
            plt.show()
else:
    print('No results loaded.')

## Classification Plots

In [ ]:
if not results.empty:
    clf = results[results['task_type'].eq('classification')].copy()
    if not clf.empty:
        plt.figure(figsize=(12, max(5, 0.38 * clf['label'].nunique())))
        sns.barplot(data=clf.sort_values('auroc_mean', ascending=False), y='label', x='auroc_mean', hue='task_family', orient='h')
        plt.axvline(0.5, color='black', lw=0.8, ls='--')
        plt.title('baseline body-composition classification AUROC by feature set')
        plt.xlabel('Mean fold AUROC')
        plt.ylabel('')
        save_fig(FIG_DIR / 'baseline_fat_classification_auroc_by_feature_set.png')
        plt.show()

        plt.figure(figsize=(12, max(5, 0.38 * clf['label'].nunique())))
        sns.barplot(data=clf.sort_values('auprc_mean', ascending=False), y='label', x='auprc_mean', hue='task_family', orient='h')
        plt.title('baseline body-composition classification AUPRC by feature set')
        plt.xlabel('Mean fold AUPRC')
        plt.ylabel('')
        save_fig(FIG_DIR / 'baseline_fat_classification_auprc_by_feature_set.png')
        plt.show()

        binary_tasks = clf[clf['task_family'].isin(['best_denovo_cardiometabolic_threshold', 'clinical_high_fat', 'exploratory_low_ffm'])]['task_id'].unique()
        for task_id in binary_tasks:
            arms_for_task = [a for a in ARM_ORDER if a in set(oof_predictions[oof_predictions['task_id'].eq(task_id)]['arm'])]
            plt.figure(figsize=(8, 6))
            for arm_name in arms_for_task:
                p = oof_predictions[(oof_predictions['task_id'].eq(task_id)) & (oof_predictions['arm'].eq(arm_name))].dropna(subset=['y_score'])
                if p.empty or p['y_true'].nunique() < 2:
                    continue
                fpr, tpr, _ = roc_curve(p['y_true'].astype(int), p['y_score'])
                row = clf[(clf['task_id'].eq(task_id)) & (clf['arm'].eq(arm_name))]
                auc = row['auroc_mean'].iloc[0] if not row.empty else roc_auc_score(p['y_true'].astype(int), p['y_score'])
                plt.plot(fpr, tpr, lw=2, label=f'{ARM_LABELS.get(arm_name, arm_name)} (AUC={auc:.3f})')
            plt.plot([0, 1], [0, 1], '--', color='black', lw=1)
            plt.xlabel('False positive rate')
            plt.ylabel('True positive rate')
            plt.title(f'ROC: {task_id}')
            plt.legend(fontsize=8)
            save_fig(FIG_DIR / f'baseline_fat_roc_{task_id}.png')
            plt.show()

            plt.figure(figsize=(8, 6))
            for arm_name in arms_for_task:
                p = oof_predictions[(oof_predictions['task_id'].eq(task_id)) & (oof_predictions['arm'].eq(arm_name))].dropna(subset=['y_score'])
                if p.empty or p['y_true'].nunique() < 2:
                    continue
                precision, recall, _ = precision_recall_curve(p['y_true'].astype(int), p['y_score'])
                row = clf[(clf['task_id'].eq(task_id)) & (clf['arm'].eq(arm_name))]
                ap = row['auprc_mean'].iloc[0] if not row.empty else average_precision_score(p['y_true'].astype(int), p['y_score'])
                plt.plot(recall, precision, lw=2, label=f'{ARM_LABELS.get(arm_name, arm_name)} (AUPRC={ap:.3f})')
            prevalence = oof_predictions[oof_predictions['task_id'].eq(task_id)]['y_true'].mean()
            plt.axhline(prevalence, color='black', lw=1, ls='--', label=f'Prevalence={prevalence:.3f}')
            plt.xlabel('Recall')
            plt.ylabel('Precision')
            plt.title(f'Precision-recall: {task_id}')
            plt.legend(fontsize=8)
            save_fig(FIG_DIR / f'baseline_fat_pr_{task_id}.png')
            plt.show()
else:
    print('No results loaded.')

## Threshold Scan Plot

In [ ]:
if 'threshold_scan' in globals() and not threshold_scan.empty:
    plt.figure(figsize=(9, 5))
    sns.lineplot(data=threshold_scan, x='quantile', y='selection_auroc', hue='target_label', marker='o')
    plt.title('Moving-threshold selection using De novo cardiometabolic')
    plt.xlabel('Threshold quantile')
    plt.ylabel('AUROC in De novo cardiometabolic')
    save_fig(FIG_DIR / 'baseline_fat_moving_threshold_scan_denovo_cardiometabolic.png')
    plt.show()
    display(threshold_scan.sort_values(['target_name', 'selection_auroc'], ascending=[True, False]).head(40))
else:
    print('No threshold scan table loaded.')

## Supplementary Random Forest Feature Importance

In [ ]:
rf_results_path = OUT_DIR / 'supplementary_random_forest_baseline_fat_results.csv'
rf_importance_path = OUT_DIR / 'supplementary_random_forest_baseline_fat_feature_importance.csv'


def feature_names_from_preprocessor(fitted_pipeline, original_X):
    pre = fitted_pipeline.named_steps['pre']
    try:
        return pre.get_feature_names_out().tolist()
    except Exception:
        names = []
        for name, transformer, cols in pre.transformers_:
            if name == 'remainder' or transformer == 'drop':
                continue
            if name == 'num':
                names.extend([f'num__{c}' for c in cols])
            elif name == 'cat':
                try:
                    onehot = transformer.named_steps['onehot']
                    names.extend(onehot.get_feature_names_out(cols).tolist())
                except Exception:
                    names.extend([f'cat__{c}' for c in cols])
        return names


def fit_rf_importance(arm, y_table, task_type):
    merged = arm['x'].merge(y_table, on='participant_id', how='inner').dropna(subset=['target_value'])
    y = merged['target_value'].astype(int) if task_type == 'classification' else pd.to_numeric(merged['target_value'], errors='coerce')
    X = merged.drop(columns=['participant_id', 'target_value'])
    est = pipeline_for(X, task_type, model_name='random_forest')
    est.fit(X, y)
    model = est.named_steps['model']
    importances = getattr(model, 'feature_importances_', None)
    if importances is None:
        return pd.DataFrame()
    names = feature_names_from_preprocessor(est, X)
    if len(names) != len(importances):
        names = [f'feature_{i}' for i in range(len(importances))]
    return pd.DataFrame({'arm': arm['arm'], 'label': arm['label'], 'feature': names, 'importance': importances}).sort_values('importance', ascending=False)

if RUN_TRAINING and RUN_RF_SUPPLEMENT and not results.empty:
    rf_rows = []
    imp_rows = []
    arm_lookup = {a['arm']: a for a in arms}
    for task in task_rows:
        if task['task_family'] not in ['continuous', 'best_denovo_cardiometabolic_threshold', 'clinical_high_fat', 'exploratory_low_ffm']:
            continue
        task_res = results[results['task_id'].eq(task['task_id'])].copy()
        best_alt = task_res[~task_res['arm'].isin(['age_sex_only', 'paper_basic_nutrients', 'nutrimatch_all'])].sort_values('primary_metric', ascending=False).head(1)
        compare = ['nutrimatch_all'] + (best_alt['arm'].tolist() if not best_alt.empty else [])
        for arm_name in compare:
            arm = arm_lookup[arm_name]
            print('RF supplement:', task['task_id'], arm_name, flush=True)
            metrics, _ = evaluate_arm(arm, task['data'], task['task_type'], model_name='random_forest')
            if metrics is not None:
                row = {k: v for k, v in task.items() if k != 'data'}
                row.update(metrics)
                rf_rows.append(row)
            imp = fit_rf_importance(arm, task['data'], task['task_type'])
            if not imp.empty:
                imp['target_name'] = task['target_name']
                imp['target_label'] = task['target_label']
                imp['task_id'] = task['task_id']
                imp['task_family'] = task['task_family']
                imp_rows.append(imp)
    rf_results = add_primary_metric(pd.DataFrame(rf_rows)) if rf_rows else pd.DataFrame()
    rf_importance = pd.concat(imp_rows, ignore_index=True) if imp_rows else pd.DataFrame()
    rf_results.to_csv(rf_results_path, index=False)
    rf_importance.to_csv(rf_importance_path, index=False)
    print('Wrote:', rf_results_path)
    print('Wrote:', rf_importance_path)
elif rf_results_path.exists():
    rf_results = pd.read_csv(rf_results_path, low_memory=False)
    rf_importance = pd.read_csv(rf_importance_path, low_memory=False) if rf_importance_path.exists() else pd.DataFrame()
    print('Loaded saved RF supplement:', rf_results_path, rf_results.shape)
else:
    rf_results = pd.DataFrame()
    rf_importance = pd.DataFrame()
    print('No RF supplement loaded.')

if not rf_importance.empty:
    for task_id in rf_importance['task_id'].drop_duplicates().head(6):
        p0 = rf_importance[rf_importance['task_id'].eq(task_id)]
        arms_plot = p0['arm'].drop_duplicates().tolist()
        fig, axes = plt.subplots(len(arms_plot), 1, figsize=(12, 5 * len(arms_plot)))
        if not isinstance(axes, np.ndarray):
            axes = np.array([axes])
        for ax, arm_name in zip(axes, arms_plot):
            p = p0[p0['arm'].eq(arm_name)].sort_values('importance', ascending=False).head(25).copy()
            p['feature_short'] = p['feature'].astype(str).str.replace(r'^(num|cat)__', '', regex=True).str.slice(0, 80)
            sns.barplot(data=p, y='feature_short', x='importance', ax=ax, color='#4C72B0')
            ax.set_title(f'{task_id}: {ARM_LABELS.get(arm_name, arm_name)}')
            ax.set_xlabel('Random Forest impurity importance')
            ax.set_ylabel('')
        save_fig(FIG_DIR / f'supplementary_rf_feature_importance_{task_id}.png')
        plt.show()
else:
    print('No RF feature importance table loaded.')